## Integrating LangChain Community Tools

Pydantic AI is modular. If there is a tool that already exists in the Langchain ecosystem, you can easily wrap it and use it with your Pydantic AI agent.

To demonstrate, let's use the real LangChain Wikipedia tool!

In [1]:
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

# Langchain tool setup
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
langchain_wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

## Adapting it to Pydantic AI

We simply write a @agent.tool_plain wrapper that invokes the Langchain .invoke() method inside!

In [2]:
import nest_asyncio
nest_asyncio.apply()

from pydantic_ai import Agent
from dotenv import load_dotenv
load_dotenv()

research_agent = Agent(
    'groq:llama-3.1-8b-instant', 
    system_prompt=( 
        "You are a smart researcher. Search Wikipedia if you don't know the answer. "
        "Be extremely careful to format tool calls exactly as requested without outputting extra raw strings."
    )
)

@research_agent.tool_plain
def search_wikipedia(search_string: str) -> str:
    """
    Use this to search Wikipedia for factual information. 
    Provide the specific topic name as the search_string.
    """
    print(f"\n[System] Reaching out to Wikipedia for: '{search_string}'...")
    # We just run the langchain tool from inside our Pydantic tool!
    return langchain_wiki_tool.invoke({"query": search_string})

In [3]:
print("--- Asking the Agent a History Question ---")
response = research_agent.run_sync("What year was the James Webb Space Telescope launched? Summarize briefly.")

print("\n--- Final Answer ---")
print(response.output)

--- Asking the Agent a History Question ---

[System] Reaching out to Wikipedia for: 'James Webb Space Telescope launch date'...

[System] Reaching out to Wikipedia for: 'James Webb Space Telescope launch'...

[System] Reaching out to Wikipedia for: 'James Webb Space Telescope'...

[System] Reaching out to Wikipedia for: 'James Webb Space Telescope launch date'...

--- Final Answer ---
The James Webb Space Telescope was launched on December 25th, 2021.
